# Six studies agree. The seventh is in pounds.

Pooling is the cheapest evidence anyone will ever buy: six studies of the same thing, one
number, a standard error smaller than any of them. It is also where the most embarrassing
mistakes live, and they are all the same mistake — pooling things that are not the same thing.
An elasticity and a per-kilogram response. A study whose "effect" was measured over four weeks
and one over twelve. Two reads by the same team, counted as independent evidence.

None of those show up in the arithmetic. The pooled number comes out fine, with a tighter
interval than any input, which is exactly what makes it convincing.

`axiom.meta` pools *dimensionless* estimates of one quantity across studies. Everything starts
from a **`StudyRecord`**: one study's estimate of one poolable quantity, its standard error, and
its provenance — who produced it (`contributor`), whether it is a fitted model's read or a
randomized experiment's read (`read`), and the pooling family it belongs to. A **`Corpus`** is an
ordered set of records with unique study ids.

This notebook covers the record schema and the catalog of poolable quantities, the ingest gate
(`normalize`) that refuses what cannot be pooled, the content-addressed `CorpusStore`, and the
sampler-free classical estimators: fixed effect, random effects with three `tau²` estimators,
heterogeneity statistics, Knapp–Hartung intervals, and the prediction interval.

In [ ]:
import tempfile

import numpy as np
import pandas as pd

from axiom.core import Assumption, D, Interval, Spec, Summary, Unsupported, dimensionless
from axiom.estimands import EstimandResult, TransferPlan
from axiom.meta import (
    DEFAULT_COLUMNS, POOLABLE_QUANTITIES, Corpus, CorpusStore, Heterogeneity, PoolableQuantity,
    PooledEstimate, ReadKind, StudyRecord, TauEstimate, TauMethod, fixed_effect, from_frame,
    heterogeneity, normalize, poolable_quantity, prediction_interval, random_effects,
    record_from_result, record_from_summary, reml_log_likelihood, se_from_summary,
    tau_dersimonian_laird, tau_paule_mandel, tau_reml,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, intervals

enable();  # every axiom result renders itself from here on

## The catalog of poolable quantities

`POOLABLE_QUANTITIES` names what may be pooled at all. Each `PoolableQuantity` says whether the
quantity is dimensionless by construction and on which scale it is pooled: `"log"` quantities
(a response ratio, an odds ratio) are strictly positive and are pooled as `log(estimate)` with
the delta-method se. The two catalog entries that carry units (`value_ratio`,
`cost_per_outcome`) can only enter a pool through a licensed transfer.

In [ ]:
table(
    [[q.name, str(q.dimensionless), q.scale, q.description] for q in POOLABLE_QUANTITIES.values()],
    headers=("quantity", "dimensionless", "scale", "description"),
)
entry: PoolableQuantity = poolable_quantity("elasticity")
print("\nlookup:", entry)
try:
    poolable_quantity("lift")
except ValueError as e:
    print("unknown quantity:", str(e)[:60], "...")

## `StudyRecord` and `Corpus`

Six studies of the dose elasticity of an outcome in the `fertilizer` family, from three
contributors. `read` is a `ReadKind` (`"model"` or `"experiment"`); `moderators` are
study-level covariates for meta-regression. A record with a non-dimensionless `dimension`
cannot be built without declaring its `unit_scale`; the ingest gate then demands a transfer.

In [ ]:
Y = [0.42, 0.55, 0.31, 0.67, 0.48, 0.39]
SE = [0.10, 0.15, 0.12, 0.20, 0.11, 0.14]


def study(i: int, read: ReadKind) -> StudyRecord:
    return StudyRecord(
        study=f"s{i}", contributor=f"c{i % 3}", quantity="elasticity", estimate=Y[i], se=SE[i],
        read=read, family="fertilizer", n=40 + 10 * i, moderators={"follow_up": float(4 + 2 * i)},
        source=f"report-{i}",
    )


records = tuple(study(i, "experiment" if i % 2 else "model") for i in range(6))
corpus = Corpus(records=records, name="demo")
print("k =", len(corpus), "| families:", corpus.families(), "| contributors:", corpus.contributors())
print("dual-read contributors (both reads in one family):", corpus.dual_read_contributors("fertilizer"))
print("round-trips:", Spec.from_json(corpus.to_json()) == corpus, "| hash:", corpus.content_hash()[:12])
corpus.to_frame()

In [ ]:
try:
    StudyRecord(study="bad", contributor="c", quantity="elasticity", estimate=1.0, se=0.1,
                read="model", family="fertilizer", dimension=D.outcome)
except ValueError as e:
    print("refused at construction:", e)
sub = corpus.by_family("fertilizer")
y, se = sub.arrays()
print(sub.name, "| arrays:", y, se)

## Ingest: `from_frame`, `normalize`, and the refusals

`from_frame` builds records from a pandas frame under an explicit field → column mapping
(`DEFAULT_COLUMNS`, overridable entry by entry); nothing is inferred from column names.
`normalize` is the one gate every record passes before pooling. It returns a `Corpus` or a typed
`Unsupported` naming the study and what would admit it. It never coerces.

In [ ]:
frame = pd.DataFrame({
    "id": ["a1", "a2", "a3"], "who": ["lab-1", "lab-2", "lab-1"], "quantity": "elasticity",
    "est": [0.5, 0.6, 0.4], "stderr": [0.1, 0.2, 0.15], "read": ["experiment", "model", "experiment"],
    "family": "fertilizer", "weeks": [4.0, 8.0, 6.0],
})
print("default mapping:", DEFAULT_COLUMNS)
recs = from_frame(frame, columns={"study": "id", "contributor": "who", "estimate": "est", "se": "stderr"},
                  moderators=["weeks"])
print(recs[0].study, recs[0].contributor, recs[0].moderators)
admitted = normalize(frame, columns={"study": "id", "contributor": "who", "estimate": "est", "se": "stderr"},
                     moderators=["weeks"], name="from-frame")
print(type(admitted).__name__, len(admitted), admitted.name)

### Refused: a record on a unit scale, without a plan

Two studies report the same shape per kilogram and per pound. Pooling is over dimensionless
quantities, so `normalize` refuses them — the reason names the study and `missing` names the
`licensed_transfer_plan` that would admit it.

In [ ]:
per_kg = records[0].model_copy(update={"unit_scale": "per_kg"})
per_lb = records[1].model_copy(update={"unit_scale": "per_lb"})
refused = normalize([per_kg, per_lb])
assert isinstance(refused, Unsupported)
print(refused.reason)
print("missing:", refused.missing)

### Accepted: the same records with a licensed `TransferPlan`

A `TransferPlan` whose `status` is `identified` or `downgraded` is *licensed*. With a plan per
study the records are admitted, and each carries the transfer in its `detail` — the plan's
content hash, status and assumptions — so the pool's provenance shows which numbers crossed a
scale boundary. A plan for only one of the two studies still refuses the other.

In [ ]:
plan = TransferPlan(status="identified", source="per_kg", target="dimensionless", differing=(),
                    entries=(), assumptions=(), ledger_lines=())
print("licensed:", plan.licensed)
partial = normalize([per_kg, per_lb], plans={"s0": plan})
print("one plan:", type(partial).__name__, "-", partial.reason[:40], "...")
licensed = normalize([per_kg, per_lb], plans={"s0": plan, "s1": plan}, name="licensed")
assert isinstance(licensed, Corpus)
table(
    [[r.study, r.detail["transfer"], r.detail["transfer_plan_hash"][:10]] for r in licensed.records],
    headers=("study", "transfer", "plan hash"),
)

In [ ]:
ratio = record_from_summary(study="neg", contributor="c9", quantity="response_ratio", estimate=-0.2,
                            se=0.1, read="experiment", family="fertilizer")
print(normalize([ratio]).reason)
print(normalize([records[0], records[0]]).reason)

## Emitting records: `record_from_summary` and `record_from_result`

`record_from_summary` is for hand-entered numbers (a published table). `record_from_result`
turns an `estimands.EstimandResult` into a record: the dimension and estimand hash come from the
result, and the se from its posterior `Summary` via `se_from_summary` — exactly
`width / (2z)` for a `wald` or `eti` interval, the posterior sd otherwise. `detail["se_from"]`
records which rule produced the number. A dimensioned result gets its unit as `unit_scale`, so
the record is constructible and `normalize` can demand a transfer.

In [ ]:
wald_summary = Summary(mean=0.52, median=0.52, sd=0.08, interval=Interval(lower=0.3632, upper=0.6768, definition="wald", mass=0.95), n=4000)
hdi_summary = Summary(mean=0.52, median=0.50, sd=0.08, interval=Interval(lower=0.36, upper=0.66, definition="hdi", mass=0.9), n=4000)
print(se_from_summary(wald_summary))
print(se_from_summary(hdi_summary))

result = EstimandResult(
    estimand_hash="a" * 64, estimand_name="elasticity_at_100", kind="elasticity", summary=wald_summary,
    dimension=dimensionless(), unit="", status="downgraded", n_draws=4000, ledger=(),
    assumptions=(Assumption(name="local_linearity", facet="intervention",
                            statement="the elasticity is read at dose 100 and holds nearby",
                            challenged_by="a second dose level"),),
    producer_hash="b" * 64,
)
rec = record_from_result(result, study="fit-1", contributor="c0", read="model", family="fertilizer")
print(rec.quantity, rec.estimate, round(rec.se, 4), rec.detail)

## `CorpusStore`

A corpus is a spec like any other, so it lives in an `io.ArtifactRegistry` as
`root/<hash>.json`. `put` is idempotent by content hash; `load_corpus(name)` returns the most
recently stored corpus of that name.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    store = CorpusStore(tmp)
    digest = store.put_corpus(corpus)
    again = store.put(corpus)
    print("idempotent:", digest == again, "| artifacts:", len(store), "| corpora:", store.corpora())
    print("kinds:", sorted({t for _, t in store.list()}))
    print("records stored:", len(store.list("StudyRecord")), "| loaded equal:", store.load_corpus("demo") == corpus)
    print("one record back:", store.get_record(store.list("StudyRecord")[0][0]).study)

## Fixed effect and heterogeneity

With `v_i = se_i²` and `w_i = 1/v_i`: `μ̂ = Σ w_i y_i / Σ w_i`, `se = (Σ w_i)^{-1/2}`, and the
interval is `μ̂ ± z·se` (a `core.Interval` labelled `wald`). `heterogeneity` reports Cochran's
`Q` about the fixed-effect mean on `k − 1` df, its chi-square `p`, `I² = max(0, (Q − df)/Q)` and
`H² = Q/df`.

In [ ]:
fe: PooledEstimate = fixed_effect(y, se)
print(f"fixed effect: {fe.estimate:.4f} ± {fe.se:.4f}  {fe.interval}")
print("weights sum to one:", round(sum(fe.weights), 12), "| quantile:", fe.detail["quantile"])
het: Heterogeneity = heterogeneity(y, se)
print(f"Q = {het.q:.3f} on {het.df} df, p = {het.p_value:.3f}, I² = {het.i2:.3f}, H² = {het.h2:.3f}")

In [ ]:
from axiom.meta import forest_data
from axiom.viz import forest

forest(forest_data(y, se, fe, labels=[r.study for r in corpus.records]))

## Three `tau²` estimators

Each returns a `TauEstimate` with the record of how it was found: DerSimonian–Laird is the
closed form `(Q − df) / (Σw − Σw²/Σw)` truncated at zero; Paule–Mandel solves
`Q(τ²) = k − 1` and REML the restricted score equation, both by `brentq` to `xtol = 1e−12`
with a boundary solution at zero when the criterion is already non-positive there.
`reml_log_likelihood` lets you check that the REML root is a maximum.

In [ ]:
estimates = {m: f(y, se) for m, f in (("dl", tau_dersimonian_laird), ("pm", tau_paule_mandel), ("reml", tau_reml))}
table(
    [
        [m, f"{t.tau2:.5f}", f"{t.tau:.4f}", t.iterations, str(t.converged), str(t.truncated)]
        for m, t in estimates.items()
    ],
    headers=("method", "tau²", "tau", "iterations", "converged", "truncated"),
)
t_reml: TauEstimate = estimates["reml"]
ll = reml_log_likelihood(y, se, t_reml.tau2)
print("REML log-lik at root:", round(ll, 6), "| below at ±0.01:",
      reml_log_likelihood(y, se, t_reml.tau2 + 0.01) < ll, reml_log_likelihood(y, se, max(t_reml.tau2 - 0.01, 0.0)) <= ll)

## Random effects, Knapp–Hartung, and the prediction interval

`random_effects` pools with weights `1/(v_i + τ²)` under the chosen `TauMethod`. With
`knapp_hartung=True` the se becomes `sqrt(Σ w_i*(y_i − μ̂)² / ((k − 1) Σ w_i*))` and the
quantile a `t_{k−1}` (the unmodified variant: it can narrow as well as widen). The prediction
interval `μ̂ ± t_{k−2} · sqrt(τ² + se²)` is where a *new* study's true effect is expected to
lie, and is wider than the interval for the mean whenever `τ² > 0`.

In [ ]:
rows = []
for method in ("dl", "pm", "reml"):
    tm: TauMethod = method
    re = random_effects(y, se, tau_method=tm)
    rows.append([method, f"{re.estimate:.4f}", f"{re.se:.4f}", f"{re.tau2:.5f}", str(re.interval)])
table(rows, headers=("tau method", "mu", "se", "tau²", "interval"))
kh = random_effects(y, se, tau_method="reml", knapp_hartung=True)
print(f"\nKnapp–Hartung: {kh.estimate:.4f} ± {kh.se:.4f}  quantile {kh.detail['quantile']}  {kh.interval}")
pi = prediction_interval(kh)
print("prediction interval:", pi, "| wider than the mean's:", pi.width > kh.interval.width)
print("tau estimate carried:", kh.tau_estimate.method, "| round-trips:", Spec.from_json(kh.to_json()) == kh)

In [ ]:
# The same six studies, and a version where one of them disagrees.
y_split = np.array(y, dtype=float).copy()
y_split[3] = 1.10
rows = []
for label, values in (("as reported", np.asarray(y, dtype=float)), ("with one study at 1.10", y_split)):
    f_ = fixed_effect(values, se)
    r_ = random_effects(values, se, tau_method="reml", knapp_hartung=True)
    p_ = prediction_interval(r_)
    h_ = heterogeneity(values, se)
    rows += [
        (f"{label} · fixed effect  (I² {h_.i2:.0%})", f_.estimate, f_.interval.lower, f_.interval.upper),
        (f"{label} · random effects", r_.estimate, r_.interval.lower, r_.interval.upper),
        (f"{label} · prediction interval", r_.estimate, p_.lower, p_.upper),
    ]

fig = intervals(
    rows,
    highlight="with one study at 1.10 · prediction interval",
    title="What heterogeneity does to the answer",
    subtitle="the same six standard errors, with and without one study that disagrees",
    x_title="elasticity",
)
caption(fig, "In the top block the studies agree, τ² is zero, and all three intervals "
             "coincide. Move one study and the fixed-effect interval barely widens — it is "
             "not allowed to — while the prediction interval, the one that answers 'where "
             "will the next study land', is nearly five times wider. Quoting the first when you mean "
             "the second is the most common way a meta-analysis overstates what is known.")

## What this bought you

A record that carries what was measured, by whom, and how it was read; an ingest gate that
refuses a per-kilogram number rather than averaging it in; three τ² estimators that each report
whether they converged and whether they were truncated at zero; and a prediction interval next
to the mean's, so a pooled result cannot be read as narrower than the evidence is.

`02-bayesian-pool.ipynb` fits the same structure hierarchically, which is where partial pooling
and the provenance offset live.